# Phase W2 — CovariateMeshRanders on Sim2Real-Fire (3D Triangular Mesh)

**Scientific question:** Does explicitly encoding DEM-derived 3D terrain geometry into the Finsler metric improve arrival-time correlation compared to the flat-grid W1 baseline, particularly for high-slope (rugged) fires?

**Key difference from W1:** Instead of treating the landscape as a flat 2D grid, Phase W2 builds a **triangular mesh** directly from the Digital Elevation Model. The metric `F(x, v)` operates in the face tangent plane of this 3D surface.

**Reference:** Gahtan, Shpund & Bronstein (2026). *Wildfire Simulation with Differentiable Randers-Finsler Eikonal Solvers.* arXiv:2603.00035.

---

## Expected results by slope level

| Slope level | Expected W1 r | Expected W2 r | Key factor |
|-------------|--------------|--------------|-----------|
| Flat        | ~0.85        | ~0.83        | Mesh adds overhead, minimal gain |
| Moderate    | ~0.75        | ~0.80        | Slope channels add some signal |
| Rugged      | ~0.60        | ~0.72        | 3D geometry captures slope-driven spread |

---

## Runtime guide

| Mode | Data needed | ~Time (T4 GPU) |
|------|------------|----------------|
| `QUICK = True` + `USE_SYNTHETIC = True` | none | ~8 min |
| `QUICK = False` + `USE_SYNTHETIC = True` | none | ~45 min |
| `QUICK = False` + `USE_SYNTHETIC = False` | Sim2Real-Fire dataset | 3–10 h per scene |

In [ ]:
# ============================================================
# CELL 1 — Colab environment setup
# ============================================================
import subprocess


def _run(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stderr[-1500:])
    return r.returncode == 0


print("Installing JAX (CUDA 12)...")
_run("pip install -q --upgrade 'jax[cuda12]'")
print("Installing HAMTools...")
_run("pip install -q git+https://github.com/hubibala/ham.git")
print("Installing wildfire dependencies...")
_run("pip install -q 'Pillow>=9.0' 'rasterio>=1.3'")
print("Done. Restart runtime if JAX was upgraded.")

In [ ]:
# ============================================================
# CELL 2 — GPU configuration
# ============================================================
import jax

try:
    from ham.utils import configure_device
except ImportError:

    def configure_device(device: str):
        dev = jax.devices(device)[0]
        jax.config.update("jax_default_device", dev)
        print(f"[HAMTools inline] Default JAX device set to: {dev}")
        return dev


devices = jax.devices()
print(f"Available devices: {devices}")
DEVICE = "gpu" if any("gpu" in str(d).lower() for d in devices) else "cpu"
configure_device(DEVICE)
print(f"Using: {DEVICE}")

In [ ]:
# ============================================================
# CELL 3 — Configuration
# ============================================================
QUICK = True  # smoke test; set False for full experiment
USE_SYNTHETIC = True  # set False + provide DATA_ROOT for real data
USE_WIND = True  # False = Riemannian ablation (b=0)
OUTPUT_DIR = "results/phaseW2"

# Real data — uncomment and configure if USE_SYNTHETIC = False
# from google.colab import drive; drive.mount('/content/drive')
# DATA_ROOT  = '/content/drive/MyDrive/sim2real_fire'
# SCENE_IDS  = ['0014_00426']
DATA_ROOT = "data/sim2real_fire"
SCENE_IDS = ["0001"]

import os

os.makedirs(f"{OUTPUT_DIR}/figs", exist_ok=True)
print(f"Config: QUICK={QUICK}, USE_SYNTHETIC={USE_SYNTHETIC}, DEVICE={DEVICE}")

In [ ]:
# ============================================================
# CELL 4 — Imports
# ============================================================
import sys

import matplotlib
import numpy as np

matplotlib.use("Agg")
import jax
import matplotlib.pyplot as plt
from jax import config

config.update("jax_enable_x64", True)

# Import from experiment script
sys.path.insert(0, "../")  # adjust to '/content/HAM/examples' on Colab
try:
    from experiment_wildfire_mesh import (
        _make_synthetic_scenario_mesh,
        build_scene_mesh,
        get_config,
        make_mesh_metric,
        make_solver,
        pearson_r,
        run_experiment_mesh,
        run_synthetic_mesh,
    )

    print("Imported experiment_wildfire_mesh OK")
except ImportError as e:
    print(f"Import error: {e}")
    print("Make sure examples/ is in sys.path.")

In [ ]:
# ============================================================
# CELL 5 — Configuration object with GPU tuning
# ============================================================
cfg = get_config(quick=QUICK)

if DEVICE == "gpu":
    cfg["batch_size_fires"] = 16 if not QUICK else 8
    if not QUICK:
        cfg["mesh_resolution_fraction"] = 1.0

print("Configuration:")
for k, v in cfg.items():
    print(f"  {k:35s} = {v}")

In [ ]:
# ============================================================
# CELL 6 — Visualise mesh construction
# ============================================================
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

print("Building example mesh from a synthetic moderate-slope scenario...")
demo_scenario = _make_synthetic_scenario_mesh(slope_level="moderate", seed=42)
demo_mesh, demo_cov, demo_fuel, demo_elev_m = build_scene_mesh(
    demo_scenario, mesh_resolution_fraction=cfg.get("mesh_resolution_fraction", 0.5)
)

print(f"Mesh: {demo_mesh.vertices.shape[0]} vertices, {demo_mesh.faces.shape[0]} faces")
print(
    f"Vertex Z range: [{demo_mesh.vertices[:, 2].min():.1f}, "
    f"{demo_mesh.vertices[:, 2].max():.1f}] m"
)

fig = plt.figure(figsize=(11, 4))

ax1 = fig.add_subplot(121, projection="3d")
sc = ax1.scatter(
    demo_mesh.vertices[:, 0],
    demo_mesh.vertices[:, 1],
    demo_mesh.vertices[:, 2],
    c=demo_mesh.vertices[:, 2],
    cmap="terrain",
    s=6,
)
plt.colorbar(sc, ax=ax1, label="Elevation (m)", shrink=0.6)
ax1.set_title("Mesh vertices — moderate slope (3D)")
ax1.set_xlabel("X (m)")
ax1.set_ylabel("Y (m)")
ax1.set_zlabel("Z (m)")

from ham.utils.terrain import compute_face_slopes_aspects

face_slopes_rad, _ = compute_face_slopes_aspects(demo_mesh)
face_slopes_deg = np.degrees(np.array(face_slopes_rad))
ax2 = fig.add_subplot(122)
ax2.hist(face_slopes_deg, bins=20, color="#4477AA", edgecolor="white")
ax2.set_xlabel("Face slope (°)")
ax2.set_ylabel("Count")
ax2.set_title(f"Slope distribution (std={face_slopes_deg.std():.1f}°)")

plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/figs/w2_mesh_demo.png", dpi=120)
plt.show()

In [ ]:
# ============================================================
# CELL 7 — Run experiment
# ============================================================
if USE_SYNTHETIC:
    print(
        "Running synthetic W2 experiment (3 slope levels: flat / moderate / rugged)..."
    )
    all_results = run_synthetic_mesh(cfg, output_dir=OUTPUT_DIR, use_wind=USE_WIND)
else:
    print(f"Running real-data W2 on scenes: {SCENE_IDS}")
    all_results = run_experiment_mesh(
        data_root=DATA_ROOT,
        scene_ids=SCENE_IDS,
        output_dir=OUTPUT_DIR,
        cfg=cfg,
        use_wind=USE_WIND,
    )

print("\n=== Results Summary ===")
for r in all_results:
    key_field = r.get("slope_level", r.get("scene_id", "?"))
    print(
        f"  {key_field:20s}  r={r['test_pearson_r_mean']:.4f}  "
        f"RMSE={r.get('test_rmse', float('nan')):.4f}  "
        f"slope_std={r.get('slope_std_deg', 0):.1f}°"
    )

In [ ]:
# ============================================================
# CELL 8 — Slope-stratified results (synthetic)
# ============================================================
if USE_SYNTHETIC and all_results:
    levels = [r.get("slope_level", str(i)) for i, r in enumerate(all_results)]
    r_vals = [r["test_pearson_r_mean"] for r in all_results]
    rmse_vals = [r.get("test_rmse", 0) for r in all_results]
    slope_stds = [r.get("slope_std_deg", 0) for r in all_results]

    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    colors = ["#5599DD", "#EE7733", "#CC3311"]

    axes[0].bar(levels, r_vals, color=colors, edgecolor="white")
    axes[0].axhline(0.70, color="red", linestyle="--", label="Target r=0.70")
    axes[0].set_ylim(0, 1.05)
    axes[0].set_ylabel("Pearson r")
    axes[0].set_title("Pearson r by slope level")
    axes[0].legend()

    axes[1].bar(levels, rmse_vals, color=colors, edgecolor="white")
    axes[1].set_ylabel("RMSE")
    axes[1].set_title("RMSE by slope level")

    axes[2].bar(levels, slope_stds, color=colors, edgecolor="white")
    axes[2].set_ylabel("Slope std (°)")
    axes[2].set_title("Terrain ruggedness")

    plt.suptitle("Phase W2 — Slope-stratified performance", fontsize=12)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/figs/w2_slope_stratified.png", dpi=120)
    plt.show()

In [ ]:
# ============================================================
# CELL 9 — Loss convergence per slope level
# ============================================================
n = len(all_results)
fig, axes = plt.subplots(1, n, figsize=(5 * n, 4), squeeze=False)
for ax, r in zip(axes[0], all_results):
    label = r.get("slope_level", r.get("scene_id", "?"))
    epochs = np.arange(1, len(r["train_loss_history"]) + 1)
    ax.plot(epochs, r["train_loss_history"], color="#4477AA")
    ax.set_xlabel("Epoch")
    ax.set_ylabel("MSE loss")
    ax.set_title(f"{label}\nr={r['test_pearson_r_mean']:.4f}")
plt.suptitle("Phase W2 — Loss convergence per slope level", fontsize=11)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/figs/w2_loss_convergence.png", dpi=120)
plt.show()

In [ ]:
# ============================================================
# CELL 10 — Predicted vs GT scatter (first result)
# ============================================================
if all_results and "pred" in all_results[0]:
    r = all_results[0]
    pred = np.array(r["pred"])
    gt = np.array(r["gt"])
    level = r.get("slope_level", r.get("scene_id", "result"))

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(gt, pred, s=10, alpha=0.5, color="#4477AA")
    lim = max(gt.max(), pred.max())
    ax.plot([0, lim], [0, lim], "r--", linewidth=1, label="Perfect")
    ax.set_xlabel("Ground-truth arrival time (normalised)")
    ax.set_ylabel("Predicted geodesic arc length (normalised)")
    ax.set_title(f"W2 {level}: r={r['test_pearson_r_mean']:.4f}")
    ax.legend()
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/figs/w2_scatter_{level}.png", dpi=120)
    plt.show()
else:
    print(
        "No 'pred'/'gt' keys in results — scatter plot requires run_synthetic_mesh to export them."
    )

In [ ]:
# ============================================================
# CELL 11 — Checkpoint note
# ============================================================
import os

print(f"Checkpoints from run_experiment_mesh are saved automatically to: {OUTPUT_DIR}/")
print("Files saved:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    if f.endswith(".eqx") or f.endswith(".json"):
        print(f"  {OUTPUT_DIR}/{f}")

# Copy to Google Drive on Colab:
# import shutil, glob
# for ckpt in glob.glob(f"{OUTPUT_DIR}/*.eqx"):
#     shutil.copy(ckpt, '/content/drive/MyDrive/ham_checkpoints/')